In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from lsst.rsp import RSPDiscovery

RA, DEC = 225.6712047, -40.2819795
DIA_OBJECT_ID = 743575318450667695
BAND = "r"

# Connect to DP2
service = RSPDiscovery("dp2").get_tap_client()

# Query forced photometry
query = f"""
SELECT
    fsodo.band,
    fsodo.psfFlux,
    fsodo.psfFluxErr,
    fsodo.psfFlux_flag,
    fsodo.pixelFlags_bad,
    fsodo.pixelFlags_saturatedCenter,
    fsodo.invalidPsfFlag,
    vis.expMidptMJD
FROM dp2.ForcedSourceOnDiaObject AS fsodo
JOIN dp2.Visit AS vis
    ON vis.visit = fsodo.visit
WHERE fsodo.diaObjectId = {DIA_OBJECT_ID}
"""

job = service.submit_job(query)
job.run()
job.wait(phases=["COMPLETED", "ERROR"])

tab = job.fetch_result().to_table()

# Save data
tab.write(f"dp2_forced_{DIA_OBJECT_ID}.ecsv", overwrite=True)

ok = (
    (np.array(tab["band"], dtype=str) == BAND)
    & (tab["psfFlux_flag"] == 0)
    & (tab["pixelFlags_bad"] == 0)
    & (tab["pixelFlags_saturatedCenter"] == 0)
    & (tab["invalidPsfFlag"] == 0)
)

mjd = np.array(tab["expMidptMJD"], dtype=float)
flux = np.array(tab["psfFlux"], dtype=float) / 1e3
flux_err = np.array(tab["psfFluxErr"], dtype=float) / 1e3

ok &= np.isfinite(mjd) & np.isfinite(flux) & np.isfinite(flux_err) & (flux_err > 0)

order = np.argsort(mjd[ok])

t = mjd[ok][order]
y = flux[ok][order]
sigma = flux_err[ok][order]

# Relative time
t -= np.floor(t.min())



$$\{t_i,\ y_i,\ \sigma_i\}_{i=1}^{N}$$

Each visit $i$ gives a time $t_i$ (days since the first night), a flux $y_i$ ($\mu Jy$) and its 1$\sigma$ error $\sigma_i$. Rubin measures the flux at the star's fixed position on every image ("forced photometry"). We are gonna use one band.

In [ ]:
plt.figure(figsize=(10, 4))
plt.errorbar(t, y, yerr=sigma, fmt="ko", ms=3)

N = t.size

plt.xlabel("Time [days]")
plt.ylabel(r"Flux [$\mu$Jy]")
plt.title(f"DiaObject {DIA_OBJECT_ID} — {BAND}-band")
plt.tight_layout()
plt.show()

$$t_i \in \text{one night}$$

Plot the night with the most visits (the star changes over a few hours).

In [ ]:
night = np.cumsum(np.r_[0, np.diff(t) > 0.4])      
on = night == np.bincount(night).argmax()          

plt.errorbar(24*(t[on] - t[on][0]), y[on], sigma[on], fmt='ko', ms=3)
plt.xlabel('hours')
plt.ylabel('flux [mJy]')
plt.show()

$$y_i = b + m\,t_i + \varepsilon_i,\qquad \varepsilon_i\sim\mathcal N(0, \sigma_i^2)$$

$$\mathbf y = A\mathbf w + \boldsymbol\varepsilon,\qquad A = \begin{bmatrix} 1 & t_1\\ \vdots & \vdots\\ 1 & t_N\end{bmatrix},\qquad \mathbf w = \begin{bmatrix} b\\ m\end{bmatrix},\qquad C = \mathrm{diag}(\sigma_1^2, \ldots, \sigma_N^2)$$

$$\chi^2(\mathbf w) = (\mathbf y - A\mathbf w)^\top C^{-1}(\mathbf y - A\mathbf w),\qquad \hat{\mathbf w} = (A^\top C^{-1}A)^{-1}A^\top C^{-1}\mathbf y,\qquad \Sigma_w = (A^\top C^{-1}A)^{-1}$$

Each flux is a straight line plus Gaussian noise of known size $\sigma_i$. $N$ is the number of visits, the columns of $A$ are the shapes the model is built from (a design matrix), and $\mathbf w$ holds their amplitudes. Minimising $\chi^2$ gives $\hat{\mathbf w}$; $\Sigma_w$ is its covariance (the error bars on $b$ and $m$). Write `wls` and fit the line.

In [ ]:
def wls(A, y, err):
    # 1/sigma^2 for every visit (the diagonal of C^{-1})
    Ci = 1 / err**2
    # F = A^T C^{-1} A
    F = A.T @ (A * Ci[:, None])
    # w_hat = F^{-1} A^T C^{-1} y   (use np.linalg.solve, not inv)
    w = np.linalg.solve(F, A.T @ (Ci * y))
    # Sigma_w = F^{-1}
    Sigma_w = np.linalg.inv(F)
    # chi^2 of the best fit
    chi2 = np.sum(Ci * (y - A @ w)**2)
    return w, Sigma_w, chi2

def line_design(x):
    # design matrix of a straight line: columns 1 and x
    return np.column_stack([np.ones_like(x), x])

w1, Sigma1, chi2_1 = wls(line_design(t), y, sigma)
print('b, m =', w1)

plt.errorbar(t, y, sigma, fmt='ko', ms=3)
plt.plot(t, line_design(t) @ w1)
plt.xlabel('t [d]'); 
plt.ylabel('flux [mJy]')
plt.show()

$$r_i = y_i - (A\hat{\mathbf w})_i,\qquad \frac{r_i}{\sigma_i}\sim\mathcal N(0, 1)\ \text{ if the model is right},\qquad \chi^2 \approx N - 2$$

$r_i$ is what the model misses, i.e. the residual. If the model and the error bars are right, each $r_i/\sigma_i$ is a draw from a unit Gaussian, so most lie within ±2 and $\chi^2$ is about the number of data minus the number of weights. Is the line a good model?

In [ ]:
r1 = y - line_design(t) @ w1                              # residuals of the line
print('chi2 / (N − 2) =', chi2_1 / (N - 2))

plt.plot(t, r1/sigma, 'ko', ms=3)
plt.xlabel('t [d]')
plt.ylabel(r'$r / \sigma$')
plt.show()

$$C = \mathrm{diag}(\sigma_i^2 + s^2),\qquad -2\ln\mathcal L(s) = \chi^2(s) + \ln|2\pi C(s)| = \sum_i\left[\frac{r_i^2}{\sigma_i^2 + s^2} + \ln 2\pi(\sigma_i^2 + s^2)\right]$$

Maybe the error bars are too small: add an unknown extra variance $s^2$ (a "jitter") to every visit. Find the $s$ that maximises the likelihood (refit $\hat{\mathbf w}$ at each $s$). What is $s$ describing?

In [ ]:
def m2lnL(A, y, sigma, s):
    # error bars inflated by the jitter s
    err = np.sqrt(sigma**2 + s**2)
    # chi^2 of the best fit with those error bars, plus ln|2πC|
    return wls(A, y, err)[2] + np.sum(np.log(2*np.pi*err**2))

s_grid = np.geomspace(1e-3, 3, 200) * np.std(y)    # trial jitters [mJy]
curve = [m2lnL(line_design(t), y, sigma, s) for s in s_grid]
s1 = s_grid[np.argmin(curve)]                      # the s with the smallest −2 ln L

plt.semilogx(s_grid, curve)
plt.xlabel('s [mJy]'); 
plt.ylabel('−2 ln L')
plt.show()

$$corr(\Delta t) = \frac{\sum r_i\, r_j}{\sqrt{\sum r_i^2\ \sum r_j^2}},\qquad \text{over all pairs } (i, j)$$

If the misfit were independent noise, two residuals would be uncorrelated however close in time they are. $\corr(\Delta t)$ is the correlation coefficient of all pairs of visits separated by about $\Delta$: +1 means they always move together, 0 means unrelated, −1 opposite. Compute it for pairs on the same night. What does it say about the star?

You will see that $corr \sim 1$ for visits minutes apart and it falls through 0 after a few hours, whcih means that the star varies on a timescale of hours, and the "noise" is not independent. For a sinusoid $corr=\cos{2\pi\Delta/P}$, the first zero is at about P/4.

In [ ]:
I, J = np.triu_indices(N, 1)                       # every pair of visits (i < j) 
dt = t[J] - t[I]                                   # their time separations
edges = np.arange(0, 0.46, 0.02)                   # bins of Delta t, 0 to 10 hours [d]
lag = 0.5*(edges[1:] + edges[:-1])                 # bin centres

def corr_of(r):
    rho = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        # pairs whose separation falls in this bin
        m = (dt >= lo) & (dt < hi)
        # correlation coefficient of r_i and r_j over those pairs (np.nan if fewer than 20 pairs)
        rho.append(np.corrcoef(r[I[m]], r[J[m]])[0, 1] if m.sum() >= 20 else np.nan)
    return np.array(rho)

plt.plot(24*lag, corr_of(r1), 'o-')
plt.axhline(0, color='k')
plt.xlabel(r'$\Delta t$ [h]')
plt.ylabel('corr')
plt.show()

$$\begin{bmatrix} r_i\\ r_j\end{bmatrix} \sim \mathcal N\!\left(\mathbf 0,\ \sigma_r^2\begin{bmatrix} 1 & \alpha\\ \alpha & 1\end{bmatrix}\right)
\quad\Rightarrow\quad
p(r_j \mid r_i) = \mathcal N\big(\alpha\, r_i,\ \sigma_r^2(1 - \alpha^2)\big)$$

Treat two residuals less than an hour apart as a 2D Gaussian with correlation $\alpha$. Knowing $r_i$ moves the expected $r_j$ to $\alpha\,r_i$ and shrinks its spread from $\sigma_r$ to $\sigma_r\sqrt{1 - \alpha^2}$. Predict $r_j$ from $r_i$.

In [ ]:
m = (dt > 0) & (dt < 1/24)                         # pairs less than an hour apart
corr_m = np.corrcoef(r1[I[m]], r1[J[m]])[0, 1]      # their correlation
s_r = np.std(r1)                                   # spread of the residuals
mu_j = corr_m * r1[I[m]]                            # conditional mean of each r_j given its r_i
sd_j = s_r * np.sqrt(1 - corr_m**2)                 # conditional standard deviation

xs = np.linspace(r1.min(), r1.max(), 50)
plt.plot(r1[I[m]], r1[J[m]], 'ko', ms=3)
plt.plot(xs, corr_m*xs)
plt.fill_between(xs, corr_m*xs - sd_j, corr_m*xs + sd_j, alpha=0.3)
plt.xlabel('r_i')
plt.ylabel('r_j')
plt.show()

$$y_i = w_0 + w_1 s_i + w_2 s_i^2 + \varepsilon_i,\qquad s_i = \frac{t_i - \bar t}{t_{\max} - t_{\min}},\qquad A = \begin{bmatrix} 1 & s_1 & s_1^2\\ \vdots & \vdots & \vdots\\ 1 & s_N & s_N^2\end{bmatrix}$$

To build a quadratic model, you will need the same linear model with one more column in the design matrix, i.e. only A changes. $s$ is time rescaled to about −½…½ so the three columns have similar sizes. Fit the quadratic. Did it help?

In [ ]:
def quad_design(x):
    # rescaled time
    s = (x - t.mean()) / (t.max() - t.min())
    # design matrix: columns 1, s, s²
    return np.column_stack([np.ones_like(s), s, s**2])

w2, Sigma2, chi2_2 = wls(quad_design(t), y, sigma)

tt = np.linspace(t.min(), t.max(), 500)            # a fine time grid for drawing the models
plt.errorbar(t, y, sigma, fmt='ko', ms=3)
plt.plot(tt, line_design(tt) @ w1, label='line')
plt.plot(tt, quad_design(tt) @ w2, label='quadratic')
plt.legend()
plt.xlabel('t [d]')
plt.ylabel('flux [mJy]')
plt.show()

Now, let's do the real deal:
$$f(t) = w_0 + \sum_{h=1}^{H}\Big[a_h\sin\frac{2\pi h t}{P} + b_h\cos\frac{2\pi h t}{P}\Big],\qquad
A = \Big[\,1,\ \sin\tfrac{2\pi t}{P},\ \cos\tfrac{2\pi t}{P},\ \ldots,\ \sin\tfrac{2\pi H t}{P},\ \cos\tfrac{2\pi H t}{P}\,\Big]$$

Any shape that repeats every $P$ can be written as sines and cosines at periods $P, P/2, \ldots, P/H$. For a fixed $P$ it is still linear in the weights $a_h, b_h$. $a_h, b_h$ are just new columns of $A$. The number of harmonics $H$ sets how sharp a feature the model can make. Write the design matrix, columns in this order.

In [ ]:
def fourier_design(x, P, H):
    # start with the constant column
    cols = [np.ones_like(x)]
    # for each harmonic h = 1...H, add the columns sin(2 pi hx/P) and cos(2 pi hx/P)
    for h in range(1, H + 1):
        cols += [np.sin(2*np.pi*h*x/P), np.cos(2*np.pi*h*x/P)]
    # stack the columns into an n by (2H + 1) matrix
    return np.column_stack(cols)

x1 = np.linspace(0, 1, 300)
plt.plot(x1, fourier_design(x1, 1.0, 2))
plt.xlabel('t / P')
plt.show()

Beautiful! Now, for fun, let's run astropy's periodogram and plot the best period (i.e. with the lowest chi^2 on the fourier model)
$$P = 1/f_{\rm peak}$$

The Lomb–Scargle periodogram fits one sinusoid at every trial frequency $f$ and plots how much of the variance it explains. The highest peak is the best single sinusoid. Nights are a day apart, so there are false peaks ("aliases") at $f \pm 1\ \mathrm{d}^{-1}$: the cell keeps whichever strong peak a 6-harmonic Fourier fit likes best. Just run it and see.

In [ ]:
from astropy.timeseries import LombScargle
f, pw = LombScargle(t, y, sigma).autopower(minimum_frequency=0.05, maximum_frequency=20, samples_per_peak=20)
peaks = np.where((pw[1:-1] > pw[:-2]) & (pw[1:-1] > pw[2:]))[0] + 1  # local maxima of the periodogram
cands = 1 / f[peaks[np.argsort(pw[peaks])[::-1][:5]]] # periods of the 5 strongest peaks
P0 = min(np.r_[cands, 2*cands], key=lambda P: wls(fourier_design(t, P, 6), y, sigma)[2])  # the one with the lowest chi^2

plt.plot(f, pw)
plt.axvline(1/P0, color='r')
plt.xlabel('frequency [1/d]');
plt.ylabel('power')
plt.show()

$$\phi_i = \frac{t_i}{P} \bmod 1$$

$\phi_i$ is the phase. If $P$ is right, visits from months apart fall on one curve when plotted against $\phi$. Fit $H$ harmonics at $P$ and plot against phase.

In [ ]:
H = 10 # number of harmonics
w3, Sigma3, chi2_3 = wls(fourier_design(t, P0, H), y, sigma)

xx = np.linspace(0, P0, 300)  # times spanning one cycle, for drawing the model
plt.errorbar((t/P0) % 1, y, sigma, fmt='ko', ms=3)
plt.plot(xx/P0, fourier_design(xx, P0, H) @ w3)
plt.xlabel('phase')
plt.ylabel('flux [mJy]')
plt.show()

$$\frac{\chi^2}{N - n_w},\qquad n_w = \text{number of weights}$$

$\chi^2$ per degree of freedom: about 1 when a model explains the data to within the error bars, much more than 1 when it doesn't. 

I want to scan through different periods and see if fourier model can pick up the "correct" one.

In [ ]:
T = t.max() - t.min()  # time span of the data
Ps = P0 + np.linspace(-1, 1, 2001) * P0**2 / (20*T)    # trial periods around P0
chi2_P = np.array([wls(fourier_design(t, p, H), y, sigma)[2] for p in Ps])   # χ²_min of the Fourier fit at each trial period
P = Ps[np.argmin(chi2_P)] # the period with the smallest χ²
sig_P = np.ptp(Ps[chi2_P - chi2_P.min() < 1]) / 2  # half the width of the region with Δχ² < 1
print('P =', P)

near = chi2_P - chi2_P.min() < 10
plt.plot(Ps[near], chi2_P[near] - chi2_P.min())
plt.axhline(1, color='k')
plt.xlabel('P [d]')
plt.ylabel(r'$\Delta \chi^2$')
plt.show()

$$r_i = y_i - (A\hat{\mathbf w})_i,\qquad corr(\Delta t)$$

Fit the Fourier series at $P$ with its own jitter, and compare $corr(\Delta)$ of its residuals with the line's. If a model has captured the star, what is left should look like independent noise $corr \approx 0$.

In [ ]:
A3 = fourier_design(t, P, H)
s3 = s_grid[np.argmin([m2lnL(A3, y, sigma, s) for s in s_grid])]   # jitter of the Fourier model
w3, Sigma3, _ = wls(A3, y, np.sqrt(sigma**2 + s3**2))   # refit with the inflated error bars
r3 = y - A3 @ w3                                   # residuals of the Fourier model

plt.plot(24*lag, corr_of(r1), 'o-', label='line'); 
plt.plot(24*lag, corr_of(r3), 'o-', label='Fourier')
plt.axhline(0, color='k')
plt.legend()
plt.xlabel(r'$\Delta t$ [h]'); 
plt.ylabel('corr')
plt.show()

$$\hat y_* = A_*\hat{\mathbf w},\qquad \sigma_*^2 = \mathrm{diag}\big(A_*\Sigma_w A_*^\top\big) + s^2 + \sigma_i^2$$

Now, we're gonna hide every 4th night, fit each model to the rest, and predict the hidden visits. The predicted error adds three things: the uncertainty of $\hat{\mathbf w}$ propagated to the new times, the jitter $s^2$, and each hidden visit's $\sigma_i^2$. Plot predicted against observed.

In [ ]:
hid = night % 4 == 1                               # every 4th night is hidden
tr = ~hid                                          # the rest is for training

def predict(design):
    # jitter fitted on the training visits only
    s = s_grid[np.argmin([m2lnL(design(t[tr]), y[tr], sigma[tr], s) for s in s_grid])]
    # fit on the training visits, with error bars inflated by the jitter
    w, Sw, _ = wls(design(t[tr]), y[tr], np.sqrt(sigma[tr]**2 + s**2))
    # design matrix at the hidden times
    A_h = design(t[hid])
    # predicted fluxes
    mu = A_h @ w
    # predicted errors: diag(A_h Σ_w A_h^T) + s^2 + sigma_i^2
    sd = np.sqrt(np.einsum('ij,jk,ik->i', A_h, Sw, A_h) + s**2 + sigma[hid]**2)
    return mu, sd

for name, design in [('line', line_design), ('quadratic', quad_design), ('Fourier', lambda x: fourier_design(x, P, H))]:
    mu, sd = predict(design)
    plt.errorbar(y[hid], mu, sd, fmt='o', label=name)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'k')
plt.legend()
plt.xlabel('observed flux [mJy]')
plt.ylabel('predicted flux [mJy]')
plt.ylim(100,200)
plt.show()

So, the next logical question I would ask is: How many harmonics? Take many, but with a Gaussian prior on the weights that expects higher harmonics to be smaller: each step in $h$ shrinks the prior's standard deviation by $q$. Then the light curve $\mathbf f = A\mathbf w$ is Gaussian with covariance $K$, the data are $\mathbf f$ plus noise, and predicting is conditioning a Gaussian. ($\mathbf y_0$ is the flux minus its mean, so the prior can have zero mean.) Use 25 harmonics, condition on the training visits, and predict one cycle.

We say each Fourier weight is a Gaussian draw with mean zero. $\Lambda$ holds the variances of those draws, and it is diagonal because we treat the weights as independent.

$$\mathbf w \sim \mathcal N(0, \Lambda),\qquad \Lambda = \mathrm{diag}\big(\lambda_0^2,\ \lambda_1^2, \lambda_1^2,\ \lambda_2^2, \lambda_2^2,\ \ldots\big),\qquad \lambda_h = \lambda_1 q^{h-1}$$, each harmonic is expected to be q times the size of the one before.

$$\mathbf f = A\mathbf w \sim \mathcal N(0,\ A\Lambda A^\top),\qquad K = A\Lambda A^\top,\quad K_* = A\Lambda A_*^\top,\quad K_{**} = A_*\Lambda A_*^\top,\quad C = \mathrm{diag}(\sigma_i^2 + s^2)$$

$$\boldsymbol\mu_* = K_*^\top(K + C)^{-1}\mathbf y_0,\qquad \Sigma_* = K_{**} - K_*^\top(K + C)^{-1}K_*,\qquad \mathbf y_0 = \mathbf y - \bar y$$

In [ ]:
y0 = y - np.average(y, weights=1/sigma**2)         # flux minus its weighted mean
HB, q = 25, 0.6                                    # number of harmonics, shrink factor per harmonic
lam1 = np.sqrt(2) * np.std(y0)                     # prior size of the first harmonic: a sinusoid with the data's variance
Lam = np.diag(np.r_[np.std(y0), np.repeat(lam1 * q**np.arange(HB), 2)]**2)   # lambda_0^2, then lambda_h^2 twice (sin and cos)
xx = np.linspace(0, P, 300)                        # times spanning one cycle
A_tr, A_s = fourier_design(t[tr], P, HB), fourier_design(xx, P, HB)    # design matrices at the training times and on the cycle
C = np.diag(sigma[tr]**2 + s3**2)                  # noise covariance of the training visits

K = A_tr @ Lam @ A_tr.T                            # A Lambda A^T
Ks = A_tr @ Lam @ A_s.T                            # A Lambda A_*^T
Kss = A_s @ Lam @ A_s.T                            # A_* Lambda A_*^T
mu = Ks.T @ np.linalg.solve(K + C, y0[tr])         # conditional mean
sd = np.sqrt(np.clip(np.diag(Kss - Ks.T @ np.linalg.solve(K + C, Ks)), 0, None))   # conditional standard deviation

plt.errorbar((t[tr]/P) % 1, y0[tr], sigma[tr], fmt='ko', ms=3)
plt.errorbar((t[hid]/P) % 1, y0[hid], sigma[hid], fmt='ro', ms=3)
plt.plot(xx/P, mu)
plt.fill_between(xx/P, mu - 2*sd, mu + 2*sd, alpha=0.3)
plt.xlabel('phase')
plt.ylabel('flux − mean [mJy]')
plt.show()

$$\Lambda \to \infty \Rightarrow \boldsymbol\mu_* \to A_*\hat{\mathbf w}$$

Making every $\lambda$ huge removes the prior, and conditioning becomes plain least squares again. Fit the same 25 harmonics with no prior and draw both. What happens where the phases have no data?

In [ ]:
w25 = wls(A_tr, y0[tr], np.sqrt(sigma[tr]**2 + s3**2))[0]   # least-squares weights of the 25 harmonics

plt.errorbar((t[tr]/P) % 1, y0[tr], sigma[tr], fmt='ko', ms=3)
plt.plot(xx/P, mu, label='prior')
plt.plot(xx/P, A_s @ w25, label='no prior')
plt.ylim(y0.min() - 0.3*np.ptp(y0), y0.max() + 0.3*np.ptp(y0))
plt.legend()
plt.xlabel('phase')
plt.ylabel('flux − mean [mJy]')
plt.show()

$$K_{ij} = \mathbf a(t_i)^\top\Lambda\,\mathbf a(t_j) = \lambda_0^2 + \sum_{h=1}^{H}\lambda_h^2\Big[\sin\tfrac{2\pi h t_i}{P}\sin\tfrac{2\pi h t_j}{P} + \cos\tfrac{2\pi h t_i}{P}\cos\tfrac{2\pi h t_j}{P}\Big] = \lambda_0^2 + \sum_{h=1}^{H}\lambda_h^2\cos\frac{2\pi h\,(t_i - t_j)}{P} \equiv k(t_i - t_j)$$

$\mathbf a(t)$ is one row of $A$. Using $\sin a\sin b + \cos a\cos b = \cos(a - b)$, every element of $K$ depends only on the time difference $t_i - t_j$. Check this numerically, and plot $k$ against $t_i - t_j$. You will see that it depends only on $t_i - t_j$ and repeats every $P$, i.e. a periodic covariance between times. That is we never needed the weights, only the covariance$k$, which is a Gaussian process with a periodic kernel!!!

In [ ]:
def k(d):
    # prior standard deviation of each harmonic, h = 1...HB
    lam_h = lam1 * q**np.arange(HB)
    # lambda_0^2 + Sigma_h lambda_h^2 cos(2 pi h d / P), for every element of d
    h = np.arange(1, HB + 1).reshape((-1,) + (1,)*np.ndim(d))
    return np.std(y0)**2 + np.sum(lam_h.reshape(h.shape)**2 * np.cos(2*np.pi*h*d/P), axis=0)

print('max |A Lambda A^T − k(ti − tj)| =', np.abs(K - k(t[tr][:, None] - t[tr][None, :])).max())

d = np.linspace(0, 3*P, 600)
plt.plot(d, k(d))
plt.xlabel('t − t` [d]')
plt.ylabel('k')
plt.show()